# Learn2Branch Set Cover 100k Colab Run

Use **Runtime > Change runtime type > T4 GPU** before running.

Run cells in order. If Colab disconnects or the runtime dies, come back here and run from **Cell 2** onward. The runner restores the newest partial sample checkpoint from Drive automatically.

Drive folder used for saved files: `/content/drive/MyDrive/learn2branch_setcover_100k`


In [ ]:
# Cell 1: Install Conda
# This intentionally restarts the runtime. After the restart, continue at Cell 2.
!pip -q install condacolab
import condacolab
condacolab.install()


In [ ]:
%%bash
# Cell 2: Install runtime environment
# Real work uses /usr/local/bin/python because Colab's visible notebook Python can stay on 3.12.
set -euo pipefail
find /usr/local -path '*/conda-meta/pinned' -type f -delete || true
conda config --remove-key pinned_packages || true
conda install -y -c conda-forge python=3.11 ecole pyscipopt numpy scipy pandas matplotlib
/usr/local/bin/python -m pip install -U pip
/usr/local/bin/python -m pip install torch torch-geometric
/usr/local/bin/python - <<'PY'
import ecole, torch, torch_geometric, pyscipopt, sys
print('python:', sys.executable, sys.version)
print('ecole:', ecole.__version__)
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('torch_geometric:', torch_geometric.__version__)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
PY


In [ ]:
# Cell 3: Mount Drive and show saved state
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/learn2branch_setcover_100k
!ls -lh /content/drive/MyDrive/learn2branch_setcover_100k


In [ ]:
%%bash
# Cell 4: Check Drive checkpoint state
set -euo pipefail
DRIVE=/content/drive/MyDrive/learn2branch_setcover_100k
echo "Drive folder: $DRIVE"
ls -lh "$DRIVE" || true
echo
echo "Partial checkpoint sample counts:"
cat "$DRIVE"/partial_samples_*_count.txt 2>/dev/null || true


In [ ]:
# Cell 5: Run or resume the full pipeline with streamed output
# This downloads the latest runner from GitHub and runs it inside Colab.
import os, subprocess

drive = "/content/drive/MyDrive/learn2branch_setcover_100k"
os.makedirs(drive, exist_ok=True)

print("downloading runner", flush=True)
subprocess.run([
    "curl", "-fL",
    "https://raw.githubusercontent.com/skanda-vyas-srinivasan/Branch-Learning-Distance-Threshold/main/colab_learn2branch_runner.py",
    "-o", "/content/colab_learn2branch_runner.py",
], check=True)

print("runner downloaded", flush=True)
print(open("/content/colab_learn2branch_runner.py").read().splitlines()[0:5], flush=True)

log_path = os.path.join(drive, "runner.log")
print("starting runner", flush=True)

with open(log_path, "a", buffering=1) as log:
    p = subprocess.Popen(
        ["/usr/local/bin/python", "-u", "/content/colab_learn2branch_runner.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in p.stdout:
        print(line, end="", flush=True)
        log.write(line)
    rc = p.wait()

print("runner exited", rc, flush=True)
if rc:
    raise RuntimeError(f"runner failed with exit code {rc}")


In [ ]:
%%bash
# Cell 6: Check progress without starting anything new
set -euo pipefail
DRIVE=/content/drive/MyDrive/learn2branch_setcover_100k
SAMPLES=/content/learn2branch-ecole/data/samples/setcover/500r_1000c_0.05d
printf 'train samples: '
find "$SAMPLES/train" -maxdepth 1 -name 'sample_*.pkl' 2>/dev/null | wc -l
printf 'valid samples: '
find "$SAMPLES/valid" -maxdepth 1 -name 'sample_*.pkl' 2>/dev/null | wc -l
echo
echo 'Drive artifacts:'
ls -lh "$DRIVE"
echo
echo 'Last runner log lines:'
tail -n 60 "$DRIVE/runner.log" 2>/dev/null || true


In [ ]:
%%bash
# Cell 7: Emergency checkpoint current local samples to Drive
# Run this before leaving or disconnecting if Cell 5 is NOT currently running.
set -euo pipefail
DRIVE=/content/drive/MyDrive/learn2branch_setcover_100k
SRC=/content/learn2branch-ecole/data/samples/setcover/500r_1000c_0.05d
STAMP=$(date +%Y%m%d_%H%M%S)
mkdir -p "$DRIVE"
echo "checkpoint start $STAMP"
find "$SRC" -name 'sample_*.pkl' 2>/dev/null | wc -l | tee "$DRIVE/partial_samples_${STAMP}_count.txt"
tar -C /content/learn2branch-ecole/data/samples/setcover -czf "$DRIVE/partial_samples_500r_1000c_0.05d_${STAMP}.tar.gz" 500r_1000c_0.05d
ls -lh "$DRIVE"
echo "checkpoint done $STAMP"
